In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# Hill function
def hill(x,A,B,C,N):
    x = np.clip(x,1e-6,1e6)
    return A*(x**N)/(B**N+x**N)+C


H2O2_params = {
"AT":[0.836384942,0.001085287,0.020947926,0.893406228],
"GC":[0.793063901,0.003510244,0.07686459,0.936160936],
"TG":[0.855612178,0.000794612,0.070094087,0.758826963],
"TCTA":[0.773519649,0.001507455,0.155108873,0.557479242],
}

OH_params = {
"AT":[0.916990085,0.000253656,0.069155176,1.732128602],
"GC":[0.901283169,0.000319827,1.20754e-21,1.866347581],
"TG":[0.855692132,5.37807e-06,0.087536377,2.431551544],
"TCTA":[0.947735022,2.65811e-05,3.49404e-21,1.539953359],
}

NO3_params = {
"AT":[0.673375337,0.00502307,1.62487e-17,13.57349854],
"GC":[0.913292006,0.003669045,1.94032e-23,1.661215552],
"TG":[0.928907612,0.004846066,2.51132e-21,1.20922578],
"TCTA":[0.65500641,0.0071487052,1.10953e-17,7.309164656],
}

NO2_params = {
"AT":[0.854516206,6.62637e-07,2.23838e-23,0.649132913],
"GC":[0.849766466,6.80344e-06,2.26173e-19,2.155228561],
"TG":[0.644761024,1.46385e-06,0.223077273,1.768813238],
"TCTA":[0.910793206,1.96472e-05,2.31943e-21,1.349954466],
}

param_sets=[H2O2_params,OH_params,NO3_params,NO2_params]

seqs=("AT","GC","TCTA","TG")
species=["H2O2","OH","NO3","NO2"]

R_data=[
[0.6546717498907749,0.85091885,0.61206873,0.79226236],
[0.6609275575102155,0.86903445,0.61919204,0.81043352],
[0.7007114027391309,0.88569822,0.63290086,0.82977677]
]

solutions=[]

for i in range(3):

    R_target=R_data[i]

    def objective(x):

        x_sum=np.sum(x)
        error=0

        for j in range(4):

            seq=seqs[j]
            Ri_pred=0

            for k in range(4):
                A,B,C,N=param_sets[k][seq]
                Ri_pred+=(x[k]/x_sum)*hill(x[k],A,B,C,N)

            error+=(Ri_pred-R_target[j])**2

        return error


    bounds=[(0.0001,0.01),(0.000001,0.001),(0.00001,0.1),(0.000001,0.01)]
    x0=[0.001,0.0001,0.01,0.0001]

    constraints={'type':'ineq','fun':lambda x: x[2]-x[3]}

    result=minimize(objective,x0=x0,bounds=bounds,constraints=constraints,method="SLSQP")

    sol=result.x
    solutions.append(sol)

    print("\nScenario",i+1)
    print("H2O2 =",sol[0],"M")
    print("OH   =",sol[1],"M")
    print("NO3  =",sol[2],"M")
    print("NO2  =",sol[3],"M")


solutions=np.array(solutions)

H2O2=solutions[:,0]
OH=solutions[:,1]
NO3=solutions[:,2]
NO2=solutions[:,3]

# Unit conversion
H2O2_mM = H2O2*1000
NO3_mM  = NO3*1000
OH_uM   = OH*1e6
NO2_uM  = NO2*1e6

scenarios=["45 W","50 W","55 W"]

fig,axs=plt.subplots(2,2,figsize=(6,5))

axs[0,0].plot(scenarios,H2O2_mM,'o-',color='black')
axs[0,0].set_title("H2O2")
axs[0,0].set_ylabel("Concentration (mM)")

axs[0,1].plot(scenarios,NO3_mM,'o-',color='black')
axs[0,1].set_title("NO3")
axs[0,1].set_ylabel("Concentration (mM)")

axs[1,0].plot(scenarios,OH_uM,'o-',color='black')
axs[1,0].set_title("OH")
axs[1,0].set_ylabel("Concentration (µM)")
axs[1,0].set_xlabel("Scenario")

axs[1,1].plot(scenarios,NO2_uM,'o-',color='black')
axs[1,1].set_title("NO2")
axs[1,1].set_ylabel("Concentration (µM)")
axs[1,1].set_xlabel("Scenario")

plt.tight_layout()
plt.show()